# Lesson 14 – Reward Diagnostics and Reward Hacking

Chapter 7 improves GRPO by tracking metrics, detecting instability, preventing reward exploitation, and adding format rewards.

For Weird AI, we are not running full GRPO training yet. Instead, this lesson asks: **Are our rewards safe and useful enough to train against?**

## 1. Why Diagnostics Matter

A model can improve its reward score while producing worse outputs for humans. For Weird AI, it might repeat rhyming words, make every line extremely short, ignore the parody topic, or include explanation text instead of lyrics.

In [ ]:
sample_rewards = [0.2, 0.3, 0.5, 0.7, 0.72, 0.75, 0.74, 0.73]
for step, reward in enumerate(sample_rewards, start=1):
    print(f'Step {step}: reward={reward:.2f}')

## 2. Moving Averages

Training metrics are noisy. A moving average smooths values so trends are easier to see.

In [ ]:
def moving_average(values, window_size=3):
    smoothed = []
    for i in range(len(values)):
        start = max(0, i - window_size + 1)
        window = values[start:i + 1]
        smoothed.append(sum(window) / len(window))
    return smoothed

for raw, smooth in zip(sample_rewards, moving_average(sample_rewards)):
    print(f'raw={raw:.2f}, moving average={smooth:.2f}')

## 3. Reward Collapse

If every rollout gets the same reward, there is no useful relative signal.

```text
Rewards:    [0.5, 0.5, 0.5, 0.5]
Advantages: [0.0, 0.0, 0.0, 0.0]
```

In [ ]:
def advantages(rewards):
    avg = sum(rewards) / len(rewards)
    return [reward - avg for reward in rewards]

print('Collapsed:', advantages([0.5, 0.5, 0.5, 0.5]))
print('Healthy:', advantages([0.1, 0.4, 0.7, 0.9]))

## 4. Advantage Standard Deviation

The advantage average should be close to zero. The standard deviation is more useful: near zero means weak or collapsed signal; moderate means useful signal; very large may mean unstable signal.

In [ ]:
import statistics
for label, rewards in [('collapsed', [0.5,0.5,0.5,0.5]), ('healthy', [0.1,0.4,0.7,0.9])]:
    adv = advantages(rewards)
    print(label, adv, 'std=', round(statistics.pstdev(adv), 3))

## 5. Reward Hacking

Reward hacking happens when a model satisfies the reward function without satisfying the real goal.

In [ ]:
reward_hacked = '''
night
fight
light
right
'''
print(reward_hacked)
print('This may rhyme, but is it a good parody?')

## 6. Repetition as a Warning Sign

A little repetition can be musical. Too much repetition can be reward hacking.

In [ ]:
def repeated_line_ratio(text):
    lines = [line.strip().lower() for line in text.splitlines() if line.strip()]
    seen = set(); repeated = 0
    for line in lines:
        if line in seen:
            repeated += 1
        else:
            seen.add(line)
    return repeated / max(1, len(lines))

lyrics = '''
night night
night night
night night
query in the moonlight
'''
print('Repeated line ratio:', repeated_line_ratio(lyrics))

## 7. Short High-Reward Outputs

A very short output with a high reward can be suspicious.

In [ ]:
short_output = 'night\nfight'
reward = 0.95
line_count = len([line for line in short_output.splitlines() if line.strip()])
print('Line count:', line_count)
print('Reward:', reward)
if reward > 0.85 and line_count < 4:
    print('Warning: suspiciously short high-reward output.')

## 8. Format Rewards

A format reward might check non-empty output, target line count, reasonable line lengths, and no explanation text.

In [ ]:
def simple_format_reward(text, target_line_count=8, tolerance=2):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    score = 0.0
    if lines: score += 0.25
    if abs(len(lines) - target_line_count) <= tolerance: score += 0.25
    if lines and all(5 <= len(line) <= 100 for line in lines): score += 0.25
    lower = text.lower()
    if 'here are' not in lower and 'explanation:' not in lower: score += 0.25
    return score

print(simple_format_reward('Here are the lyrics:\nnight\nfight', target_line_count=4))

## 9. Combining Creative and Format Rewards

A format reward should nudge the model toward usable outputs without replacing the creative score.

In [ ]:
creative_reward = 0.82
format_reward = 0.60
combined = 0.75 * creative_reward + 0.25 * format_reward
print(round(combined, 3))

## 10. Reflection

1. Why can reward increase while human-perceived quality decreases?
2. What does reward collapse mean?
3. Why is advantage standard deviation useful?
4. How can format rewards help Weird AI?
5. How can format rewards accidentally hurt Weird AI?
6. What reward hacking behavior would you watch for first?